# Lab 7 — Local Engineering Knowledge Copilot

End-to-end modular RAG and agent application built with **Redis Stack**, **Sentence-Transformers**, **OpenAI Agents SDK**, **FastAPI**, **Streamlit**, and **LM Studio**.

This notebook walks through the live runtime modules in `labs/lab07_final_project/src/knowledge_copilot/`:
- Markdown/text extraction and structure-aware chunking over `assets/knowledge/`
- In-memory `BM25Retriever` lexical scoring and `reciprocal_rank_fusion` (RRF)
- Safe AST arithmetic tool (`safe_calculate`), `SQLiteRepository` session isolation, and `GroundingVerifier`
- Optional live `QueryService` turn when Redis Stack and LM Studio are running (`LAB07_RUN_LIVE=1`)

## 1. Environment & Paths

In [1]:
import os, sys, json, hashlib, tempfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "labs/lab07_final_project/src").is_dir())
LAB_ROOT = REPO_ROOT / "labs/lab07_final_project"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Lab Root:", LAB_ROOT.relative_to(REPO_ROOT).as_posix())
print("Pandas:", pd.__version__)

Lab Root: labs/lab07_final_project
Pandas: 2.3.3


## 2. Domain Models & Grounding Status Contracts

In [2]:
from labs.lab07_final_project.src.knowledge_copilot.domain.enums import (
    DocumentStatus, EvidenceQuality, RetrievalMode, GroundingStatus, ScoreType
)
from labs.lab07_final_project.src.knowledge_copilot.domain.models import (
    Document, Chunk, Score, EvidenceChunk, Turn
)

print("Document statuses:", [s.value for s in DocumentStatus])
print("Evidence quality levels:", [q.value for q in EvidenceQuality])
print("Retrieval modes:", [m.value for m in RetrievalMode])
print("Grounding statuses:", [g.value for g in GroundingStatus])

Document statuses: ['PENDING', 'PROCESSING', 'INDEXED', 'FAILED']
Evidence quality levels: ['canonical_report', 'quoted_model_output', 'user_document']
Retrieval modes: ['dense', 'lexical', 'hybrid']
Grounding statuses: ['GROUNDED', 'PARTIALLY_SUPPORTED', 'INSUFFICIENT_EVIDENCE', 'VALIDATION_FAILED', 'EXECUTION_FAILED']


## 3. Knowledge Corpus Ingestion & Chunking (`assets/knowledge/`)

In [3]:
from labs.lab07_final_project.src.knowledge_copilot.rag.extractors import validate_file_content, extract_text_from_bytes
from labs.lab07_final_project.src.knowledge_copilot.rag.splitters import build_chunks_from_document

knowledge_dir = LAB_ROOT / "assets/knowledge"
source_files = sorted(p for p in knowledge_dir.rglob("*.md") if p.is_file())

corpus_chunks = []
doc_rows = []
now = datetime.now(timezone.utc).isoformat()
for md_path in source_files:
    raw_bytes = md_path.read_bytes()
    clean_name, sha_hash, size_bytes = validate_file_content(raw_bytes, md_path.name)
    text = extract_text_from_bytes(raw_bytes, clean_name)
    doc_id = md_path.stem
    doc = Document(
        document_id=doc_id, filename=clean_name, content_type="md",
        sha256=sha_hash, size_bytes=size_bytes, status=DocumentStatus.INDEXED,
        chunk_count=0, created_at=now, updated_at=now,
    )
    chunks = build_chunks_from_document(doc, text, max_tokens=220)
    corpus_chunks.extend(chunks)
    doc_rows.append({
        "document_id": doc_id,
        "path": md_path.relative_to(knowledge_dir).as_posix(),
        "size_bytes": size_bytes,
        "chunks": len(chunks),
        "sha256_prefix": sha_hash[:12],
    })

display(pd.DataFrame(doc_rows))
print(f"Indexed {len(doc_rows)} Markdown documents -> {len(corpus_chunks)} structure-aware chunks.")

.venv-lab04\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,document_id,path,size_bytes,chunks,sha256_prefix
0,adr_001_vector_and_storage_architecture,engineering/adr_001_vector_and_storage_archite...,2271,5,97319eab1244
1,api_specification_and_contracts,engineering/api_specification_and_contracts.md,2087,5,3e63e7575b20
2,incident_postmortem_redis_oom,engineering/incident_postmortem_redis_oom.md,2483,5,a6feb7f8c7e0
3,rag_system_runbook,engineering/rag_system_runbook.md,1856,4,bd348e5e76ee
4,lab03_rag_report,lab03_rag_report.md,3594,4,120a3d1b8fdf
5,lab04_stable_diffusion_report,lab04_stable_diffusion_report.md,15550,22,6b11e807a68c
6,lab05_multimodal_report,lab05_multimodal_report.md,25125,30,9c4fce1431dd
7,lab06_agents_report,lab06_agents_report.md,34221,39,bccc31c3d081


Indexed 8 Markdown documents -> 114 structure-aware chunks.


## 4. Lexical BM25 Search & Reciprocal Rank Fusion (RRF)

In [4]:
from labs.lab07_final_project.src.knowledge_copilot.rag.lexical import BM25Retriever
from labs.lab07_final_project.src.knowledge_copilot.rag.hybrid import reciprocal_rank_fusion

bm25 = BM25Retriever(k1=1.5, b=0.75)
bm25.index(corpus_chunks)

sample_queries = [
    "Which scheduler and step count were used in Stable Diffusion?",
    "What caused the Redis Stack OOM incident and what memory limit triggered it?",
    "Which vision encoder and LLM backbone are paired in LLaVA?",
]

for q in sample_queries:
    hits = bm25.search(q, k=3)
    top_info = [(c.document_id, round(score, 3), c.section[:35]) for c, score in hits]
    print(f"Query: {q}")
    print(f"  Top-3 BM25 hits: {top_info}")

# Combine simulated dense distances + live BM25 scores through RRF
lex_hits = bm25.search(sample_queries[0], k=5)
dense_sim = [(ch, 0.15 + 0.05 * idx) for idx, (ch, _) in enumerate(reversed(lex_hits))]
fused = reciprocal_rank_fusion(dense_sim, lex_hits, k=3, rrf_k=30)
display(pd.DataFrame([{
    "rank": idx, "citation": ec.citation, "rrf_score": round(ec.score.score_value, 5),
    "preview": ec.text[:100].replace("\n", " ")
} for idx, ec in enumerate(fused, 1)]))

Query: Which scheduler and step count were used in Stable Diffusion?
  Top-3 BM25 hits: [('lab04_stable_diffusion_report', 22.382, 'Лабораторна робота №4 / Модель і се'), ('lab04_stable_diffusion_report', 21.267, 'Лабораторна робота №4 / Організація'), ('lab04_stable_diffusion_report', 14.337, 'Лабораторна робота №4 / Організація')]
Query: What caused the Redis Stack OOM incident and what memory limit triggered it?
  Top-3 BM25 hits: [('incident_postmortem_redis_oom', 42.222, 'Incident Post-Mortem: Redis Out-of-'), ('incident_postmortem_redis_oom', 38.367, 'Incident Post-Mortem: Redis Out-of-'), ('incident_postmortem_redis_oom', 35.516, 'Incident Post-Mortem: Redis Out-of-')]
Query: Which vision encoder and LLM backbone are paired in LLaVA?
  Top-3 BM25 hits: [('lab04_stable_diffusion_report', 8.961, 'Лабораторна робота №4 / Теорія та а'), ('lab05_multimodal_report', 8.445, 'Лабораторна робота №5 / 6. Роль ком'), ('lab06_agents_report', 5.524, 'Лабораторна робота №6 / Теоретичні ')]


,rank,citation,rrf_score,preview
0,1,[lab04_stable_diffusion_report#chunk-aa1893557...,0.10276,## Лабораторна робота №4 / Модель і середовище...
1,2,[lab04_stable_diffusion_report#chunk-55db2fe4d...,0.10129,## Лабораторна робота №4 / Організація контрол...
2,3,[lab04_stable_diffusion_report#chunk-8362b1160...,0.10000,## Лабораторна робота №4 / Організація контрол...


## 5. Agent Orchestrator Prompts & Safe AST Arithmetic Tool

In [5]:
from labs.lab07_final_project.src.knowledge_copilot.agents.orchestrator import SYSTEM_INSTRUCTIONS
from labs.lab07_final_project.src.knowledge_copilot.agents.tools import safe_calculate

print("System instructions length:", len(SYSTEM_INSTRUCTIONS), "chars")
for expr in ["17 - 13", "2048 - 512", "(17 + 13) / 2"]:
    print(f"{expr} => {safe_calculate(expr)['result']}")

try:
    safe_calculate("__import__('os').system('echo blocked')")
except ValueError as exc:
    print("AST sandbox blocked unsafe expression:", exc)

System instructions length: 4444 chars
17 - 13 => 4
2048 - 512 => 1536
(17 + 13) / 2 => 15.0
AST sandbox blocked unsafe expression: Only numeric literals, parentheses and +, -, *, / are permitted.


## 6. Multi-Turn Session Isolation via `SQLiteRepository`

In [6]:
from labs.lab07_final_project.src.knowledge_copilot.infrastructure.sqlite_repo import SQLiteRepository

import gc
with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as tmpdir:
    repo = SQLiteRepository(db_path=Path(tmpdir) / "sessions.sqlite")
    s_a = repo.create_session("session_alpha", "Alpha User")
    s_b = repo.create_session("session_beta", "Beta User")

    repo.save_canonical_turn(Turn(
        turn_id="turn_1", session_id=s_a.session_id,
        user_input="Remember that Project Atlas targets port 6380.",
        raw_answer="Noted for session_alpha.", canonical_answer="Noted for session_alpha.",
        grounding_status=GroundingStatus.GROUNDED,
        created_at=datetime.now(timezone.utc).isoformat(), runtime_seconds=0.01,
    ))

    loaded_a = repo.get_session(s_a.session_id)
    loaded_b = repo.get_session(s_b.session_id)
    assert len(loaded_a.turns) == 1 and len(loaded_b.turns) == 0
    print(f"Session Alpha turns: {len(loaded_a.turns)} | Session Beta turns: {len(loaded_b.turns)} (isolated=True)")
    gc.collect()

Session Alpha turns: 1 | Session Beta turns: 0 (isolated=True)


## 7. Evidence Grounding & Citation Verifier

In [7]:
from labs.lab07_final_project.src.knowledge_copilot.agents.verifier import verify_grounding

top_ev = fused[0]
grounded_answer = f"Stable Diffusion v1-5 (`stable-diffusion-v1-5/stable-diffusion-v1-5`) {top_ev.citation}."
status_ok, valid_ok, invalid_ok, _ = verify_grounding(grounded_answer, fused)
assert status_ok == GroundingStatus.GROUNDED
print("Valid answer grounding:", status_ok.value, "| valid citations:", valid_ok)

fake_answer = "Stable Diffusion used 100 steps [nonexistent_doc#c_99]."
status_bad, valid_bad, invalid_bad, _ = verify_grounding(fake_answer, fused)
assert status_bad == GroundingStatus.VALIDATION_FAILED
print("Fabricated citation check:", status_bad.value, "| rejected citations:", invalid_bad)

Valid answer grounding: GROUNDED | valid citations: ['[lab04_stable_diffusion_report#chunk-aa189355741329d5]']
Fabricated citation check: VALIDATION_FAILED | rejected citations: ['[nonexistent_doc#c_99]']


## 8. Optional Live Service Probe (`LAB07_RUN_LIVE=1`)

In [8]:
if os.getenv("LAB07_RUN_LIVE", "0") == "1":
    import asyncio
    from labs.lab07_final_project.src.knowledge_copilot.api.dependencies import get_query_service
    qs = get_query_service()
    turn = asyncio.run(qs.execute_query(
        query="Which Stable Diffusion model was used in Lab 4?",
        session_id="nb07_live_demo",
    ))
    print("Grounding status:", turn.grounding_status.value)
    print("Answer:", turn.canonical_answer)
else:
    print("Offline walkthrough complete. Run FastAPI (uvicorn) or Streamlit UI, or set LAB07_RUN_LIVE=1 with Redis + LM Studio running.")

Offline walkthrough complete. Run FastAPI (uvicorn) or Streamlit UI, or set LAB07_RUN_LIVE=1 with Redis + LM Studio running.
